# Test RQ code

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys
from pathlib import Path

import pandas as pd
import seaborn as sns

#project_dir = Path.cwd().parent
#sys.path.insert(0, project_dir / 'analysis')

In [11]:
from data import load_events
from rq2_patterns import build_sequences, sequences_for

In [59]:
from collections import Counter
from collections import defaultdict

def count_ngrams(seqs: list[list[str]], n: int) -> pd.DataFrame:
    counts = Counter()
    for seq in seqs:
        for i in range(len(seq) - n + 1):
            counts[tuple(seq[i : i + n])] += 1
    return counts
    

def count_session_ngrams(seqs_by_key: dict[tuple[str, str], list[list[str]]], n: int) -> pd.DataFrame:
    counts = defaultdict(Counter)
    for part, game in seqs_by_key:
        seq = seqs_by_key[(part, game)]
        for i in range(len(seq) - n + 1):
            counts[(part, game)][tuple(seq[i : i + n])] += 1
    return counts
    

def count_participant_ngrams(seqs_by_key: dict[tuple[str, str], list[list[str]]], n: int) -> pd.DataFrame:
    counts = defaultdict(Counter)
    for part, game in seqs_by_key:
        seq = seqs_by_key[(part, game)]
        for i in range(len(seq) - n + 1):
            counts[part][tuple(seq[i : i + n])] += 1
    return counts
    

def top_ngrams(seqs: list[list[str]], n: int, k: int = 15) -> pd.DataFrame:
    top = count_ngrams(seqs, n).most_common(k)
    return pd.DataFrame(
        [{"pattern": " -> ".join(p), "count": c} for p, c in top]
    )


def top_ngram_sessions(seqs_by_key: dict[tuple[str, str], list[list[str]]], n: int, k: int = 15) -> pd.DataFrame:
    session_ngram_freq = count_session_ngrams(seqs_by_key, n)
    ngram_session_freq = Counter(seq for part, game in session_ngram_freq for seq in session_ngram_freq[(part, game)])
    top = ngram_session_freq.most_common(k)
    return pd.DataFrame(
        [{"pattern": " -> ".join(p), "count": c} for p, c in top]
    )


def top_ngram_participants(seqs_by_key: dict[tuple[str, str], list[list[str]]], n: int, k: int = 15) -> pd.DataFrame:
    participant_ngram_freq = count_participant_ngrams(seqs_by_key, n)
    ngram_participant_freq = Counter(seq for part in participant_ngram_freq for seq in participant_ngram_freq[part])
    top = ngram_participant_freq.most_common(k)
    return pd.DataFrame(
        [{"pattern": " -> ".join(p), "count": c} for p, c in top]
    )



In [60]:
events = load_events()
seqs_by_key = build_sequences(events)
seqs = sequences_for(seqs_by_key)
#seqs_by_key

In [61]:
participant_ngram_freq = count_participant_ngrams(seqs_by_key, n=2)
ngram_participant_freq = Counter(seq for part in participant_ngram_freq for seq in participant_ngram_freq[part])
ngram_participant_freq.most_common(10)

[(('Platform switch', 'Query'), 21),
 (('Query', 'View result list'), 21),
 (('View result list', 'Platform switch'), 19),
 (('View result list', 'Query'), 18),
 (('View result list', 'View single result'), 18),
 (('Query', 'Platform switch'), 15),
 (('Platform switch', 'View result list'), 15),
 (('View single result', 'Query'), 15),
 (('Query', 'View AI response'), 14),
 (('View single result', 'Experiment ended'), 13)]

In [62]:
top_ngram_participants(seqs_by_key, n=2, k=50)

,pattern,count
0,Platform switch -> Query,21
1,Query -> View result list,21
2,View result list -> Platform switch,19
3,View result list -> Query,18
4,View result list -> View single result,18
5,Query -> Platform switch,15
6,Platform switch -> View result list,15
7,View single result -> Query,15
8,Query -> View AI response,14
9,View single result -> Experiment ended,13


In [63]:
top_ngram_participants(seqs_by_key, n=3, k=50)

,pattern,count
0,Query -> View result list -> Platform switch,18
1,Platform switch -> Query -> View result list,18
2,Query -> View result list -> Query,17
3,View result list -> Query -> View result list,16
4,Query -> View result list -> View single result,16
5,View result list -> Platform switch -> Query,16
6,Platform switch -> Query -> Platform switch,14
7,View result list -> Platform switch -> View re...,13
8,Query -> Platform switch -> View result list,12
9,Platform switch -> View result list -> Platfor...,12


In [72]:
top_ngram_participants(seqs_by_key, n=4, k=10)

,pattern,count
0,Query -> View result list -> Query -> View res...,14
1,View result list -> Query -> View result list ...,12
2,Platform switch -> Query -> View result list -...,12
3,Platform switch -> Query -> Platform switch ->...,11
4,Query -> View result list -> Platform switch -...,11
5,View result list -> Platform switch -> Query -...,11
6,View result list -> Query -> View result list ...,10
7,View result list -> Platform switch -> View re...,10
8,Platform switch -> Query -> View result list -...,10
9,Query -> View result list -> Platform switch -...,10


In [70]:
seqs_by_bb_key = {(p, g): seqs_by_key[(p, g)] for p, g in seqs_by_key if g == 'Best Buds vs Bad Guys'}
len(seqs_by_bb_key), len(seqs_by_key)

(21, 42)

In [73]:
top_ngram_participants(seqs_by_bb_key, n=4, k=10)

,pattern,count
0,Query -> View result list -> Query -> View res...,13
1,View result list -> Query -> View result list ...,11
2,Query -> View result list -> Platform switch -...,10
3,Platform switch -> Query -> View result list -...,10
4,View result list -> Platform switch -> Query -...,10
5,Platform switch -> Query -> View result list -...,10
6,Platform switch -> Query -> Platform switch ->...,9
7,View result list -> Query -> View result list ...,9
8,View result list -> Platform switch -> View re...,9
9,Query -> View result list -> View single resul...,9


In [79]:
top_ngram_participants(seqs_by_bb_key, n=5, k=10)


,pattern,count
0,View result list -> Query -> View result list ...,9
1,Query -> View result list -> Platform switch -...,8
2,Query -> View result list -> Query -> View res...,8
3,Query -> View result list -> Query -> View res...,7
4,Platform switch -> Query -> View result list -...,7
5,Platform switch -> Query -> View AI response -...,7
6,Platform switch -> Query -> Platform switch ->...,6
7,Query -> View result list -> Query -> View res...,6
8,View result list -> Platform switch -> View re...,6
9,View result list -> View single result -> View...,6


In [83]:
top_ngram_participants(seqs_by_bb_key, n=6, k=10).to_dict()


{'pattern': {0: 'View result list -> Query -> View result list -> Query -> View result list -> Query',
  1: 'Query -> View result list -> Query -> View result list -> Query -> View result list',
  2: 'Query -> View result list -> Platform switch -> View result list -> Query -> View result list',
  3: 'View result list -> Query -> View result list -> Query -> View result list -> View single result',
  4: 'Platform switch -> Query -> View result list -> Platform switch -> View result list -> Query',
  5: 'Platform switch -> Query -> Platform switch -> View result list -> Query -> View result list',
  6: 'Platform switch -> View result list -> Query -> View result list -> Query -> View result list',
  7: 'View result list -> Query -> View result list -> Query -> View result list -> Platform switch',
  8: 'Query -> View result list -> Query -> View result list -> View single result -> Platform switch',
  9: 'Query -> View AI response -> Query -> View AI response -> Query -> View AI respons

In [82]:
top_ngram_participants(seqs_by_bb_key, n=7, k=10)


,pattern,count
0,View result list -> Query -> View result list ...,5
1,Platform switch -> Query -> Platform switch ->...,3
2,Query -> Platform switch -> View result list -...,3
3,Platform switch -> View result list -> Query -...,3
4,Query -> View result list -> Query -> View res...,3
5,Query -> View result list -> Query -> View res...,3
6,View result list -> Query -> View result list ...,3
7,View result list -> Query -> View result list ...,3
8,Query -> View result list -> Query -> View res...,3
9,View result list -> Query -> View result list ...,3


In [80]:
top_ngram_participants(seqs_by_bb_key, n=8, k=10)#.to_dict()

,pattern,count
0,Platform switch -> Query -> Platform switch ->...,3
1,Query -> Platform switch -> View result list -...,3
2,Platform switch -> View result list -> Query -...,3
3,View result list -> Query -> View result list ...,3
4,Query -> View result list -> Query -> View res...,3
5,View result list -> Query -> View result list ...,3
6,Query -> View result list -> Query -> View res...,3
7,Query -> View result list -> Platform switch -...,2
8,View result list -> Platform switch -> View re...,2
9,Platform switch -> View result list -> Query -...,2


In [91]:
from rq10_finer_platform_categories import load_platform_map, categorize

platform_map = load_platform_map()

In [92]:
from data import load_events


events = load_events()
events = events.copy()
is_view_single = events["action"] == "View single result"
categorizable = events["context"].notna() & ~is_view_single
events["platform_category"] = pd.NA
events.loc[categorizable, "platform_category"] = events.loc[categorizable, "context"].apply(
    lambda v: categorize(v, platform_map)
)


In [107]:
events.head(20)
events.platform_category.value_counts()
events[events.action == 'Experiment ended'].groupby('game').platform_category.value_counts()

game                   platform_category    
Best Buds vs Bad Guys  Web search                9
                       Conversational search     6
                       Game search               3
                       Image search              3
Townscaper             Image search             10
                       Conversational search     4
                       Web search                3
                       Other                     2
                       Game search               1
Name: count, dtype: int64

In [106]:
events[(events.context.isna()) ]
#events[events.participant_id == 'ID12'].tail(30)

,participant_id,rank,annotator,game,language,action,query_tactic,action_details_clean,action_details_num_tokens,step,context,platform_category
81,ID10,25,annotator2,Best Buds vs Bad Guys,EN,Query,New query,NaN,5,1,NaN,<NA>
230,ID12,55,annotator4,Townscaper,NaN,Platform switch,NaN,NaN,0,1,NaN,<NA>
231,ID12,56,annotator4,Townscaper,EN,Query,New query,NaN,4,2,NaN,<NA>
232,ID12,57,annotator4,Townscaper,NaN,View result list,NaN,NaN,2,3,NaN,<NA>
234,ID12,59,annotator4,Townscaper,NaN,View result list,NaN,NaN,2,5,NaN,<NA>
235,ID12,60,annotator4,Townscaper,NaN,View AI response,NaN,NaN,1,6,NaN,<NA>
236,ID12,61,annotator4,Townscaper,DE,Query,New query,NaN,3,7,NaN,<NA>
237,ID12,62,annotator4,Townscaper,NaN,View result list,NaN,NaN,2,8,NaN,<NA>
238,ID12,63,annotator4,Townscaper,DE,Query,New query,NaN,3,9,NaN,<NA>
239,ID12,64,annotator4,Townscaper,NaN,View result list,NaN,NaN,2,10,NaN,<NA>


In [174]:
from collections import defaultdict

from data import load_events


def count_move_context_ngrams(seqs_by_key, context_by_key, ngram_size: int = 2):
    ngram_freq = Counter()
    ngram_session_freq = defaultdict(Counter)
    ngram_user_freq = defaultdict(Counter)
    for session_id in seqs_by_key:
        user, game = session_id
        #print(session_id, len(seqs_by_key[session_id]), len(context_by_key[session_id]))
        move_contexts = [(move, context) for move, context in zip(seqs_by_key[session_id], context_by_key[session_id])]
        for i in range(len(move_contexts) - ngram_size + 1):
            selected = move_contexts[i:i+ngram_size]
            ngram_freq.update([tuple(selected)])
            ngram_user_freq[user].update([tuple(selected)])
            ngram_session_freq[session_id].update([tuple(selected)])
    return ngram_session_freq


def count_user_tactics(ngram_session_freq):
    ngrams = set(ngram for sid in ngram_session_freq for ngram in ngram_session_freq[sid])
    rows = []
    for ngram in ngrams:
        tactic = " -> ".join([f"{move} ({context})" for move, context in ngram])
        for session_id in ngram_session_freq:
            user, game = session_id
            if ngram in ngram_session_freq[session_id]:
                row = {'user': user, 'game': game, 'tactic': tactic, 'freq': ngram_session_freq[session_id].get(ngram, 0)}
                rows.append(row)
    
    return pd.DataFrame(rows)

events = load_events()
seqs_by_key = build_sequences(events)
seqs = sequences_for(seqs_by_key)


context_by_key = {(pid, game): list(grp['context']) for (pid, game), grp in events.groupby(['participant_id', 'game'])}

session_id = ('ID1', 'Best Buds vs Bad Guys')

ngram_size = 2
ngram_session_freq = count_move_context_ngrams(seqs_by_key, context_by_key, ngram_size=ngram_size)
user_game_tactic_n2_freq = count_user_tactics(ngram_session_freq)
user_game_tactic_n2_freq

,user,game,tactic,freq
0,ID10,Best Buds vs Bad Guys,View result list (Steam) -> Browsing (Steam),3
1,ID16,Townscaper,View result list (Steam) -> Browsing (Steam),1
2,ID21,Best Buds vs Bad Guys,View result list (Steam) -> Browsing (Steam),1
3,ID21,Townscaper,View result list (Steam) -> Browsing (Steam),1
4,ID17,Best Buds vs Bad Guys,View single result (souldbound.game/blog) -> Q...,1
...,...,...,...,...
770,ID1,Best Buds vs Bad Guys,View result list (Google) -> View single resul...,1
771,ID11,Townscaper,View result list (Google) -> View single resul...,1
772,ID12,Townscaper,View result list (Google) -> View single resul...,1
773,ID3,Townscaper,View single result (subreddit IndieGaming thre...,1


In [175]:
user_game_tactic_n2_freq.tactic.value_counts().head(10)

tactic
Platform switch (Google) -> Query (Google)                                         37
Query (Google) -> View result list (Google)                                        33
Platform switch (Google Image Search) -> View result list (Google Image Search)    22
View result list (Google) -> Query (Google)                                        19
Query (Google Image Search) -> View result list (Google Image Search)              19
Query (Google) -> Platform switch (Google Image Search)                            16
View result list (Google) -> Platform switch (Google Image Search)                 16
View result list (Google Image Search) -> Query (Google Image Search)              15
Query (Google) -> View AI response (Google)                                        13
View result list (Google Image Search) -> View info box (Google Image Search)      13
Name: count, dtype: int64

In [204]:
tactic = "Query (Google) -> Platform switch (Google Image Search)"
tactic_freq = user_game_tactic_n2_freq[user_game_tactic_n2_freq.tactic == tactic]
n_users = tactic_freq.user.nunique()
n_sessions = tactic_freq.user.nunique()
print(f"n_users: {n_users}, n_sessions: {n_sessions}, n_instances: {tactic_freq.freq.sum()}")
pd.concat([tactic_freq.game.value_counts().rename('sessions'), tactic_freq.groupby('game').freq.sum().rename('instances')], axis=1)

n_users: 11, n_sessions: 11, n_instances: 32


,sessions,instances
game,,
Best Buds vs Bad Guys,10,25
Townscaper,6,7


In [199]:
tactic_freq = user_game_tactic_n2_freq[user_game_tactic_n2_freq.tactic.str.endswith("Platform switch (Steam)")]
tactic_freq = tactic_freq[tactic_freq.tactic.str.contains("Google")]
n_users = tactic_freq.user.nunique()
n_sessions = tactic_freq.user.nunique()
print(f"n_users: {n_users}, n_sessions: {n_sessions}, n_instances: {tactic_freq.freq.sum()}")
tactic_freq.game.value_counts()


n_users: 2, n_sessions: 2, n_instances: 3


game
Best Buds vs Bad Guys    2
Name: count, dtype: int64

In [173]:
ngram_size = 6
ngram_session_freq = count_move_context_ngrams(seqs_by_key, context_by_key, ngram_size=ngram_size)
user_game_tactic_freq = count_user_tactics(ngram_session_freq)
user_game_tactic_freq

,user,game,tactic,freq
0,ID19,Best Buds vs Bad Guys,Query (Google) -> View result list (Google) ->...,1
1,ID12,Townscaper,Query (Google) -> View result list (Google) ->...,1
2,ID9,Best Buds vs Bad Guys,Query (Google) -> View result list (Google) ->...,1
3,ID19,Best Buds vs Bad Guys,View result list (Google) -> Platform switch (...,1
4,ID2,Best Buds vs Bad Guys,View result list (Google) -> Query (Google) ->...,1
...,...,...,...,...
1036,ID19,Best Buds vs Bad Guys,Platform switch (Google AI Mode) -> Query (Goo...,1
1037,ID9,Townscaper,View single result (Steam) -> View single resu...,6
1038,ID3,Best Buds vs Bad Guys,Browsing (Moby Games) -> Platform switch (Goog...,1
1039,ID11,Best Buds vs Bad Guys,Query (Google) -> View result list (Google) ->...,1


In [169]:
temp = user_game_tactic_freq.groupby(['game', 'tactic']).user.count().reset_index()
temp[temp.user > 2]

,game,tactic,user
68,Best Buds vs Bad Guys,Platform switch (Google Image Search) -> View ...,4
96,Best Buds vs Bad Guys,Platform switch (Google) -> Query (Google) -> ...,4
111,Best Buds vs Bad Guys,Platform switch (Google) -> Query (Google) -> ...,3
230,Best Buds vs Bad Guys,Query (Google Image Search) -> View result lis...,4
259,Best Buds vs Bad Guys,Query (Google) -> Platform switch (Google Imag...,3
503,Best Buds vs Bad Guys,View result list (Google Image Search) -> Quer...,3
504,Best Buds vs Bad Guys,View result list (Google Image Search) -> Quer...,4
